# 03-4 — MCP Catalog Adapter

Shows how to connect an MCP server and register all its tools into `AgentCatalog`
using the new `MCPCatalogAdapter` (Sprint 5).

**Modules demonstrated**:
- `ravi.integrations.mcp.adapter.MCPCatalogAdapter`
- `ravi.integrations.mcp.client.MCPClient.discover_tools()`
- `ravi.integrations.mcp.tool.MCPTool` (error handler fix)

**Prerequisites**: MCP demo server running on `localhost:9000/sse`  
```bash
docker compose -f deployment/docker/docker-compose.yml --profile mcp up -d mcp-server
```
Section 4 works offline — it shows the `MCPTool` error path without a live server.

---
## 1. Connect and discover tools

`MCPClient.discover_tools()` is the new convenience method — it calls `MCPTool.from_mcp_client()`
internally and returns `list[MCPTool]` instances ready to pass to `ReActAgent`.

In [ ]:
from ravi.integrations.mcp.client import MCPClient

MCP_URL = "http://localhost:9000/sse"

client = MCPClient()
try:
    await client.connect_sse(url=MCP_URL)
    tools = await client.discover_tools()   # ← new method (Sprint 5)
    print(f"Discovered {len(tools)} tools from {MCP_URL}:")
    for t in tools:
        print(f"  {t.name:<25} {t.description[:60]}")
except RuntimeError as e:
    print(f"[offline] Could not connect: {e}")
    tools = []

---
## 2. Register MCP tools in `AgentCatalog` via `MCPCatalogAdapter`

`MCPCatalogAdapter` wraps the discover-and-register flow into a single `await adapter.register(client)` call.  
Each tool lands in the catalog with `ResourceType.MCP_TOOL` so `catalog.get_tool()` finds it.

In [ ]:
from ravi.core.agent_catalog import AgentCatalog, ResourceType
from ravi.integrations.mcp.adapter import MCPCatalogAdapter

catalog = AgentCatalog()
adapter = MCPCatalogAdapter(catalog, namespace="demo_mcp")

if client.is_connected:
    fqns = await adapter.register(client)
    print(f"Registered {len(fqns)} tools:")
    for fqn in fqns:
        tool = catalog.get_tool(fqn.split(".", 1)[1])  # strip namespace prefix
        print(f"  {fqn}  →  ResourceType={ResourceType.MCP_TOOL.value}")
else:
    print("[offline] Skipping — no live MCP server")

---
## 3. Use registered MCP tools with `ReActAgent`

Once registered, MCP tools become standard catalog resources. Register the model,
memory, and context alongside them, then construct `ReActAgent` with the catalog as
the single source of truth.


In [ ]:
# The agent picks up all tools from the catalog automatically
all_tools = catalog.all_tools()
print(f"catalog.all_tools() → {len(all_tools)} tool(s)")

if all_tools:
    from ravi.core.agents.react_agent import ReActAgent
    from ravi.core.memory.unbounded_memory import UnboundedMemory
    from ravi.core.context.implementations import UnboundedContext
    from ravi.integrations.llm.factory import create_model_client
    import os

    CHAT_MODEL = os.environ.get("CHAT_MODEL", "openai/gpt-5.4-mini")
    API_KEYS = {
        "openai": os.environ.get("OPENAI_API_KEY", ""),
        "anthropic": os.environ.get("ANTHROPIC_API_KEY", ""),
        "google": os.environ.get("GOOGLE_API_KEY", os.environ.get("GEMINI_API_KEY", "")),
        "groq": os.environ.get("GROQ_API_KEY", os.environ.get("GROK_API_KEY", "")),
        "openrouter": os.environ.get("OPENROUTER_API_KEY", ""),
    }

    if os.environ.get("OPENAI_API_KEY"):
        if catalog.primary_model() is None:
            catalog.register_model("primary", create_model_client(CHAT_MODEL, api_keys=API_KEYS))
        if catalog.primary_memory() is None:
            catalog.register_memory("default", UnboundedMemory())
        if catalog.primary_context() is None:
            catalog.register_context("default", UnboundedContext())

        agent = ReActAgent(
            name="MCPAgent",
            description="Agent backed by MCP tools",
            catalog=catalog,
            max_iterations=5,
            verbose=True,
        )
        print(f"Agent created with {len(agent.tools)} tools from catalog")
        print(f"Configured chat model: {CHAT_MODEL}")
    else:
        print("[offline] OPENAI_API_KEY not set — skipping agent creation")
else:
    print("[offline] No tools in catalog — skipping")

---
## 4. `MCPTool` error handling fix (offline demo)

**Before Sprint 5**: the `except` block returned `ToolResult(content=[{"type": "text", ...}])` —
passing a raw `dict` where `ContentBlock` is expected. This caused a type mismatch downstream.

**After Sprint 5**: the error block uses `TextBlock(text=...)` like all other code paths.

In [ ]:
from ravi.integrations.mcp.tool import MCPTool
from ravi.core.messages.content import TextBlock
import inspect

# Verify the fix is in place
src = inspect.getsource(MCPTool.execute)
assert 'TextBlock' in src, "Error: MCPTool.execute() still uses raw dict"
# The old bad pattern should not be present
assert '{"type": "text"' not in src, "Error: raw dict still in error handler"
print("MCPTool.execute() error handler uses TextBlock — confirmed")

# Demonstrate a disconnected tool returning a proper ToolResult with TextBlock content
from unittest.mock import AsyncMock, MagicMock
mock_client = MagicMock()
mock_client.is_connected = True
mock_client.call_tool = AsyncMock(side_effect=RuntimeError("network failure"))

tool = MCPTool(
    client=mock_client,
    name="search",
    description="web search",
    input_schema={"type": "object", "properties": {}},
)

result = await tool.execute(query="test")
print(f"is_error : {result.is_error}")
print(f"content  : {result.content}")
assert isinstance(result.content[0], TextBlock), "content[0] must be a TextBlock"
print("Content block type verified: TextBlock")

---
## 5. Multiple MCP servers with different namespaces

You can register tools from multiple servers into the same catalog by using
distinct namespaces. The FQN (`namespace.name`) keeps them separate.

In [ ]:
from ravi.core.agent_catalog import AgentCatalog, ResourceSpec, ResourceType
from ravi.integrations.mcp.adapter import MCPCatalogAdapter

multi_catalog = AgentCatalog()

# Simulate two servers by manually registering specs
for server_ns, tool_names in [("filesystem", ["read_file", "write_file"]),
                               ("database",   ["query", "insert"])]:
    for tname in tool_names:
        spec = ResourceSpec(
            name=tname,
            namespace=server_ns,
            resource_type=ResourceType.MCP_TOOL,
            description=f"{tname} from {server_ns} server",
        )
        multi_catalog.register(spec, object())  # placeholder instance

all_mcp = multi_catalog.all_tools()   # returns all ResourceType.TOOL | MCP_TOOL
print(f"Total tools in catalog: {len(all_mcp)}")

# Look up by simple name (searches all namespaces)
found = multi_catalog.get_tool("query")
print(f"get_tool('query') found: {found is not None}")

In [ ]:
# Cleanup
if client.is_connected:
    await client.disconnect()
    print("Disconnected from MCP server")